In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    from_json,
    to_timestamp,
    current_timestamp,
    when,
    lit,
    year,
    month,
    dayofmonth,
    hour,
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    ArrayType,
)


# =========================================================
# CONFIG
# =========================================================

KAFKA_BOOTSTRAP_SERVERS = "YOUR_KAFKA_PRIVATE_IP:9092"
KAFKA_TOPIC = "stock-trades-test"

BRONZE_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/bronze/simulated/trades/"
)

BRONZE_CHECKPOINT_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/checkpoints/bronze_simulated_trades/"
)

QUARANTINE_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/quarantine/simulated/trades/"
)

QUARANTINE_CHECKPOINT_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/checkpoints/quarantine_simulated_trades/"
)


ALLOWED_SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
]


# =========================================================
# TRADE SCHEMA
# =========================================================

TRADE_SCHEMA = StructType([
    StructField("schema_version", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("source", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField(
        "trade_conditions",
        ArrayType(StringType()),
        True
    ),
    StructField("event_timestamp_ms", LongType(), True),
    StructField("event_timestamp_utc", StringType(), True),
    StructField("ingestion_timestamp_utc", StringType(), True),
])


def main():

    spark = (
        SparkSession.builder
        .appName("StockMarketBronzeAndQuarantine")
        .config("spark.sql.session.timeZone", "UTC")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")


    # =====================================================
    # READ FROM KAFKA
    # =====================================================

    kafka_df = (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            KAFKA_BOOTSTRAP_SERVERS
        )
        .option(
            "subscribe",
            KAFKA_TOPIC
        )
        .option(
            "startingOffsets",
            "earliest"
        )
        .option(
            "maxOffsetsPerTrigger",
            2000
        )
        .load()
    )


    # =====================================================
    # PARSE KAFKA MESSAGE
    # =====================================================

    parsed_df = (
        kafka_df
        .select(
            col("key").cast("string").alias("message_key"),
            col("value").cast("string").alias("raw_message"),
            col("topic"),
            col("partition"),
            col("offset"),
            col("timestamp").alias("kafka_timestamp"),
        )

        .withColumn(
            "parsed",
            from_json(
                col("raw_message"),
                TRADE_SCHEMA
            )
        )

        .select(
            "message_key",
            "raw_message",
            "topic",
            "partition",
            "offset",
            "kafka_timestamp",
            "parsed.*",
        )

        .withColumn(
            "event_timestamp",
            to_timestamp(
                col("event_timestamp_utc")
            )
        )

        .withColumn(
            "ingestion_timestamp",
            to_timestamp(
                col("ingestion_timestamp_utc")
            )
        )

        .withColumn(
            "processing_timestamp",
            current_timestamp()
        )
    )


    # =====================================================
    # VALIDATION
    # =====================================================

    validated_df = (
        parsed_df

        .withColumn(
            "validation_error",

            when(
                col("event_id").isNull(),
                lit("missing_event_id")
            )

            .when(
                col("symbol").isNull(),
                lit("missing_symbol")
            )

            .when(
                ~col("symbol").isin(ALLOWED_SYMBOLS),
                lit("unsupported_symbol")
            )

            .when(
                col("message_key") != col("symbol"),
                lit("key_symbol_mismatch")
            )

            .when(
                col("price").isNull()
                | (col("price") <= 0),
                lit("invalid_price")
            )

            .when(
                col("volume").isNull()
                | (col("volume") <= 0),
                lit("invalid_volume")
            )

            .when(
                col("event_timestamp").isNull(),
                lit("invalid_event_timestamp")
            )

            .when(
                col("ingestion_timestamp").isNull(),
                lit("invalid_ingestion_timestamp")
            )

            .otherwise(
                lit(None).cast("string")
            )
        )
    )


    # =====================================================
    # VALID → BRONZE
    # =====================================================

    valid_df = (
        validated_df

        .filter(
            col("validation_error").isNull()
        )

        # Bronze partitions use ingestion time
        .withColumn(
            "year",
            year(col("ingestion_timestamp"))
        )

        .withColumn(
            "month",
            month(col("ingestion_timestamp"))
        )

        .withColumn(
            "day",
            dayofmonth(col("ingestion_timestamp"))
        )

        .withColumn(
            "hour",
            hour(col("ingestion_timestamp"))
        )
    )


    # =====================================================
    # INVALID → QUARANTINE
    # =====================================================

    invalid_df = (
        validated_df

        .filter(
            col("validation_error").isNotNull()
        )

        # Use processing time here because an invalid record
        # may not have a usable ingestion timestamp.
        .withColumn(
            "year",
            year(col("processing_timestamp"))
        )

        .withColumn(
            "month",
            month(col("processing_timestamp"))
        )

        .withColumn(
            "day",
            dayofmonth(col("processing_timestamp"))
        )

        .withColumn(
            "hour",
            hour(col("processing_timestamp"))
        )
    )


    # =====================================================
    # BRONZE STREAM
    # =====================================================

    bronze_query = (
        valid_df.writeStream
        .format("parquet")
        .outputMode("append")

        .option(
            "path",
            BRONZE_PATH
        )

        .option(
            "checkpointLocation",
            BRONZE_CHECKPOINT_PATH
        )

        .option(
            "compression",
            "snappy"
        )

        .partitionBy(
            "year",
            "month",
            "day",
            "hour"
        )

        .trigger(
            processingTime="10 seconds"
        )

        .start()
    )


    # =====================================================
    # QUARANTINE STREAM
    # =====================================================

    quarantine_query = (
        invalid_df.writeStream
        .format("parquet")
        .outputMode("append")

        .option(
            "path",
            QUARANTINE_PATH
        )

        .option(
            "checkpointLocation",
            QUARANTINE_CHECKPOINT_PATH
        )

        .option(
            "compression",
            "snappy"
        )

        .partitionBy(
            "year",
            "month",
            "day",
            "hour"
        )

        .trigger(
            processingTime="10 seconds"
        )

        .start()
    )


    print("=" * 60)
    print("Bronze + Quarantine streaming writer started")
    print(f"Kafka topic: {KAFKA_TOPIC}")
    print(f"Bronze:      {BRONZE_PATH}")
    print(f"Quarantine:  {QUARANTINE_PATH}")
    print("Press Ctrl+C to stop")
    print("=" * 60)


    # Keep both streaming queries alive
    spark.streams.awaitAnyTermination()


if __name__ == "__main__":
    main()